# Advanced Poisson Regression Topics

**Chapter 6: Regression Techniques for Soccer Analytics - EXTRA**

## Learning Objectives

- Understand overdispersion in count data
- Implement Negative Binomial regression
- Handle zero-inflation in soccer data
- Build hurdle models for two-stage prediction
- Diagnose and fix Poisson model violations
- Choose the appropriate count model for your data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)


## The Problem: When Poisson Isn't Enough

**Standard Poisson assumption:** Variance = Mean

**Reality in soccer:**
- Some teams are consistently high/low scoring (extra variance)
- Many matches end 0-0 (zero-inflation)
- Goals come in clusters (overdispersion)

**Solutions:**
1. **Negative Binomial** - Handles overdispersion
2. **Zero-Inflated Poisson** - Handles excess zeros
3. **Hurdle Models** - Two-stage process


## Overdispersion: When Variance Exceeds the Mean

**Overdispersion** occurs when the variance in the data is greater than what the Poisson model expects.


In [ ]:
# Generate overdispersed data
np.random.seed(42)
n = 200

# Simulate match data with overdispersion
shots = np.random.randint(5, 15, n)
# Add team-level random effects (causes overdispersion)
team_effect = np.random.choice([-0.5, 0, 0.5], n)
goals_overdispersed = np.random.poisson(shots * 0.15 + team_effect + 1)

df_over = pd.DataFrame({
    'Shots': shots,
    'Goals': goals_overdispersed
})

print("Overdispersion Check:")
print(f"Mean goals: {df_over['Goals'].mean():.3f}")
print(f"Variance: {df_over['Goals'].var():.3f}")
print(f"Variance/Mean ratio: {df_over['Goals'].var() / df_over['Goals'].mean():.3f}")
print(f"\
If ratio > 1: Overdispersion present!")
print(f"If ratio ≈ 1: Poisson is appropriate")
print(f"If ratio < 1: Underdispersion (rare)")


## Compare Poisson vs. Negative Binomial

In [ ]:
# Fit Poisson model
poisson_model = smf.poisson('Goals ~ Shots', data=df_over).fit()

# Fit Negative Binomial model
nb_model = smf.negativebinomial('Goals ~ Shots', data=df_over).fit()

print("Model Comparison:")
print(f"\
Poisson AIC: {poisson_model.aic:.2f}")
print(f"Negative Binomial AIC: {nb_model.aic:.2f}")
print(f"\
Lower AIC = Better model")

if nb_model.aic < poisson_model.aic:
    print("✅ Negative Binomial is better - overdispersion confirmed!")
else:
    print("✅ Poisson is sufficient - no significant overdispersion")

print(f"\
Negative Binomial alpha parameter: {nb_model.params['alpha']:.3f}")
print("(Alpha > 0 indicates overdispersion)")


## Visualize the Difference

In [ ]:
# Get predictions
df_over['poisson_pred'] = poisson_model.predict(df_over)
df_over['nb_pred'] = nb_model.predict(df_over)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Actual vs Predicted - Poisson
axes[0].scatter(df_over['Goals'], df_over['poisson_pred'], alpha=0.5)
axes[0].plot([0, 8], [0, 8], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Goals')
axes[0].set_ylabel('Predicted Goals')
axes[0].set_title('Poisson Regression')
axes[0].grid(True, alpha=0.3)

# Actual vs Predicted - Negative Binomial
axes[1].scatter(df_over['Goals'], df_over['nb_pred'], alpha=0.5, color='green')
axes[1].plot([0, 8], [0, 8], 'r--', linewidth=2)
axes[1].set_xlabel('Actual Goals')
axes[1].set_ylabel('Predicted Goals')
axes[1].set_title('Negative Binomial Regression')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Negative Binomial typically provides wider prediction intervals,")
print("accounting for the extra variability in the data.")


## Zero-Inflation: Too Many Zeros

**Problem:** Some matches have more 0-0 draws than a Poisson model would predict.

**Causes in soccer:**
- Defensive tactics
- Bad weather
- Low-stakes matches
- Mismatched teams playing conservatively


In [ ]:
# Generate zero-inflated data
np.random.seed(42)
n = 200

# Some matches are "structural zeros" (defensive matches)
defensive_match = np.random.binomial(1, 0.3, n)  # 30% defensive

shots = np.random.randint(5, 15, n)
goals_zi = np.where(
    defensive_match == 1,
    0,  # Defensive matches always 0 goals
    np.random.poisson(shots * 0.15)  # Normal matches
)

df_zi = pd.DataFrame({
    'Shots': shots,
    'Goals': goals_zi
})

print("Zero-Inflation Check:")
print(f"Observed zeros: {(df_zi['Goals'] == 0).sum()} ({(df_zi['Goals'] == 0).mean()*100:.1f}%)")

# Expected zeros under Poisson
mean_goals = df_zi['Goals'].mean()
expected_zeros = len(df_zi) * np.exp(-mean_goals)
print(f"Expected zeros (Poisson): {expected_zeros:.0f} ({expected_zeros/len(df_zi)*100:.1f}%)")

if (df_zi['Goals'] == 0).sum() > expected_zeros * 1.2:
    print("\
⚠️ Zero-inflation detected! Consider Zero-Inflated Poisson (ZIP) model.")
else:
    print("\
✅ No significant zero-inflation.")


## Visualize Zero-Inflation

In [ ]:
# Compare observed vs expected distribution
fig, ax = plt.subplots(figsize=(10, 6))

# Observed distribution
observed_counts = df_zi['Goals'].value_counts().sort_index()
ax.bar(observed_counts.index - 0.2, observed_counts.values, width=0.4, 
       label='Observed', alpha=0.7, color='blue')

# Expected Poisson distribution
max_goals = df_zi['Goals'].max()
expected_counts = [len(df_zi) * stats.poisson.pmf(k, mean_goals) for k in range(max_goals + 1)]
ax.bar(np.arange(max_goals + 1) + 0.2, expected_counts, width=0.4, 
       label='Expected (Poisson)', alpha=0.7, color='orange')

ax.set_xlabel('Number of Goals')
ax.set_ylabel('Frequency')
ax.set_title('Observed vs. Expected Distribution (Zero-Inflation)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("Notice the excess of zeros in the observed data (blue bar at 0).")


## Hurdle Models: Two-Stage Approach

**Idea:** Model the process in two stages:
1. **Stage 1:** Will any goals be scored? (Binary: Yes/No)
2. **Stage 2:** If yes, how many? (Count model for positive values)

**When to use:** When the process of scoring the first goal is fundamentally different from scoring additional goals.


In [ ]:
# Implement a simple hurdle model
from sklearn.linear_model import LogisticRegression

# Stage 1: Binary model (any goals?)
df_zi['any_goals'] = (df_zi['Goals'] > 0).astype(int)
X = df_zi[['Shots']]

logit_model = LogisticRegression()
logit_model.fit(X, df_zi['any_goals'])
prob_any_goals = logit_model.predict_proba(X)[:, 1]

print("Stage 1: Probability of scoring any goals")
print(f"Average probability: {prob_any_goals.mean():.3f}")

# Stage 2: Count model for positive goals
df_positive = df_zi[df_zi['Goals'] > 0].copy()
if len(df_positive) > 0:
    poisson_positive = smf.poisson('Goals ~ Shots', data=df_positive).fit()
    print(f"\
Stage 2: Poisson model for positive goals")
    print(f"Coefficient for Shots: {poisson_positive.params['Shots']:.4f}")
    
    # Combined prediction
    positive_pred = poisson_positive.predict(df_zi[['Shots']])
    hurdle_pred = prob_any_goals * positive_pred
    
    print(f"\
Hurdle model combines both stages:")
    print(f"Prediction = P(any goals) × E(goals | goals > 0)")
else:
    print("Not enough positive observations for stage 2.")


## Goodness-of-Fit Tests

How do we know if our Poisson model fits well?


In [ ]:
# Pearson chi-square test for Poisson
poisson_simple = smf.poisson('Goals ~ Shots', data=df_over).fit()

# Calculate Pearson chi-square statistic
residuals = poisson_simple.resid_pearson
chi_square = np.sum(residuals**2)
df_resid = len(df_over) - len(poisson_simple.params)

print("Goodness-of-Fit Test:")
print(f"Pearson chi-square: {chi_square:.2f}")
print(f"Degrees of freedom: {df_resid}")
print(f"Ratio (chi²/df): {chi_square/df_resid:.2f}")
print(f"\
Interpretation:")
print(f"  Ratio ≈ 1: Good fit")
print(f"  Ratio > 1: Overdispersion (consider Negative Binomial)")
print(f"  Ratio < 1: Underdispersion (rare)")

if chi_square/df_resid > 1.5:
    print(f"\
⚠️ Model shows overdispersion. Consider Negative Binomial regression.")
elif chi_square/df_resid < 0.7:
    print(f"\
⚠️ Model shows underdispersion. Check for model misspecification.")
else:
    print(f"\
✅ Poisson model fits reasonably well.")


## Model Selection Guide

| Situation | Recommended Model |
|-----------|------------------|
| **Variance ≈ Mean** | Standard Poisson |
| **Variance > Mean** | Negative Binomial |
| **Excess zeros** | Zero-Inflated Poisson (ZIP) |
| **Two-stage process** | Hurdle Model |
| **Overdispersion + zeros** | Zero-Inflated Negative Binomial |

**Decision Tree:**
1. Check variance/mean ratio → If > 1.5, consider Negative Binomial
2. Check zero proportion → If excess, consider ZIP or Hurdle
3. Compare models using AIC/BIC → Lower is better
4. Validate with holdout data → Test generalization


## Practical Example: Choosing the Right Model

In [ ]:
# Generate realistic soccer data
np.random.seed(42)
n = 300

shots = np.random.randint(3, 18, n)
possession = np.random.uniform(35, 65, n)

# Realistic goal generation with overdispersion
team_quality = np.random.choice([0.8, 1.0, 1.2], n)
goals = np.random.negative_binomial(
    n=2,  # dispersion parameter
    p=2/(2 + shots * 0.12 * team_quality)
)

df_real = pd.DataFrame({
    'Shots': shots,
    'Possession': possession,
    'Goals': goals
})

print("Realistic Soccer Data Analysis:")
print(f"Mean: {df_real['Goals'].mean():.3f}")
print(f"Variance: {df_real['Goals'].var():.3f}")
print(f"Variance/Mean: {df_real['Goals'].var()/df_real['Goals'].mean():.3f}")
print(f"Zero proportion: {(df_real['Goals']==0).mean()*100:.1f}%")

# Fit multiple models
models = {}
models['Poisson'] = smf.poisson('Goals ~ Shots + Possession', data=df_real).fit()
models['NegBinom'] = smf.negativebinomial('Goals ~ Shots + Possession', data=df_real).fit()

print(f"\
Model Comparison:")
for name, model in models.items():
    print(f"  {name}: AIC = {model.aic:.2f}")

best_model = min(models.items(), key=lambda x: x[1].aic)
print(f"\
✅ Best model: {best_model[0]}")


## Summary

In this notebook, we:

1. ✅ Understood overdispersion and its causes
2. ✅ Implemented Negative Binomial regression
3. ✅ Detected and handled zero-inflation
4. ✅ Built hurdle models for two-stage processes
5. ✅ Performed goodness-of-fit tests
6. ✅ Learned model selection criteria

## Key Takeaways

- **Standard Poisson** assumes variance = mean (often violated in soccer)
- **Negative Binomial** handles overdispersion (variance > mean)
- **Zero-inflation** is common in soccer (defensive matches)
- **Hurdle models** separate "any goals" from "how many"
- **Always check assumptions** before choosing a model
- **AIC/BIC** help compare non-nested models
- **Real soccer data** often requires Negative Binomial

## When to Use Each Model

- **Poisson:** Quick baseline, well-behaved data
- **Negative Binomial:** Most soccer applications (handles overdispersion)
- **ZIP/Hurdle:** Defensive leagues, cup competitions with many 0-0 draws
- **ZINB:** Combination of overdispersion and zero-inflation


## Practice Exercises

1. **Implement ZIP Model**: Use `statsmodels` to fit a Zero-Inflated Poisson model
2. **Compare All Models**: Fit Poisson, NB, ZIP, and Hurdle on the same data
3. **Simulate Data**: Generate data with known properties and verify model selection
4. **Real Match Data**: Apply to actual league data and identify the best model
5. **Prediction Intervals**: Generate prediction intervals for count models
6. **Temporal Effects**: Add time trends to account for changing goal-scoring patterns
